# Chapter 25 — Bayesian Networks
*Tier 3: Data Scientists & Developers*

## 🎯 Learning Objectives
- Represent conditional independence structures with DAGs
- Perform exact inference (variable elimination)
- Build and query a real Bayesian network with pgmpy/numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import networkx as nx

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Bayesian Networks

A Bayesian network is a **DAG** where nodes are random variables
and edges encode direct conditional dependencies.

**Factorisation theorem:**
$$P(X_1, \ldots, X_n) = \prod_{i=1}^n P(X_i \mid \text{parents}(X_i))$$

This is the key computational advantage — we only need to store CPDs
(Conditional Probability Distributions), not the full joint table.

In [ ]:
# Visualise the Asia BN (classic benchmark)
G = nx.DiGraph()
edges = [
    ("Smoking", "LungCancer"),
    ("Smoking", "Bronchitis"),
    ("LungCancer", "Dyspnoea"),
    ("LungCancer", "XRay"),
    ("Bronchitis", "Dyspnoea"),
    ("VisitToAsia", "Tuberculosis"),
    ("Tuberculosis", "XRay"),
    ("Tuberculosis", "Dyspnoea"),
]
G.add_edges_from(edges)

pos = {
    "Smoking": (2, 3), "VisitToAsia": (0, 3),
    "LungCancer": (1, 2), "Bronchitis": (3, 2), "Tuberculosis": (0, 2),
    "XRay": (0.5, 1), "Dyspnoea": (2, 1),
}
plt.figure(figsize=(9, 6))
nx.draw_networkx(G, pos, node_size=2500, node_color="steelblue",
                 font_color="white", font_size=9, arrows=True, arrowsize=20,
                 connectionstyle="arc3,rad=0.05")
plt.title("Asia Bayesian Network (Classic Benchmark)")
plt.axis("off"); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Variable Elimination

To compute $P(A)$ from a joint distribution, sum out (marginalise) other variables:
$$P(A) = \sum_{B,C,\ldots} P(A, B, C, \ldots)$$

Variable elimination exploits the factorisation to do this efficiently,
multiplying and marginalising factor by factor.

In [ ]:
# Manual Bayesian network inference: Burglar Alarm example
# P(B), P(E), P(A|B,E), P(J|A), P(M|A)

p_B = 0.001  # prior burglary
p_E = 0.002  # prior earthquake

# P(A | B, E)
p_A_given = {
    (True,  True):  0.95,
    (True,  False): 0.94,
    (False, True):  0.29,
    (False, False): 0.001,
}
p_J_given = {True: 0.90, False: 0.05}  # P(J=1 | A)
p_M_given = {True: 0.70, False: 0.01}  # P(M=1 | A)

def compute_joint(b, e, a, j, m):
    p_b = p_B if b else 1-p_B
    p_e = p_E if e else 1-p_E
    p_a = p_A_given[(b,e)] if a else 1-p_A_given[(b,e)]
    p_j = p_J_given[a] if j else 1-p_J_given[a]
    p_m = p_M_given[a] if m else 1-p_M_given[a]
    return p_b * p_e * p_a * p_j * p_m

# Query: P(B=1 | J=1, M=1)
numerator   = sum(compute_joint(True,  e, a, True, True)
                  for e in [True,False] for a in [True,False])
denominator = sum(compute_joint(b, e, a, True, True)
                  for b in [True,False] for e in [True,False] for a in [True,False])
print(f"P(Burglary | JohnCalls=1, MaryCalls=1) = {numerator/denominator:.6f}")

In [ ]:
# Naive Bayes classifier (a simple BN)
from sklearn.datasets import load_iris
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X, y = iris.data, iris.target
nb_model = GaussianNB()
scores = cross_val_score(nb_model, X, y, cv=10)
print(f"Naive Bayes on Iris — CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

nb_model.fit(X, y)
print("
Class priors:", nb_model.class_prior_.round(3))
print("Learned feature means per class:")
print(nb_model.theta_.round(2))

In [ ]:
# Practice: build a simple spam filter BN
# P(spam), P(word | spam), P(word | not spam)
p_spam = 0.30
# Feature: contains word "free"
p_free_given_spam = 0.70
p_free_given_ham  = 0.05

# Query: P(spam | "free" present)
p_free = p_free_given_spam * p_spam + p_free_given_ham * (1 - p_spam)
p_spam_given_free = (p_free_given_spam * p_spam) / p_free
print(f"P(spam | 'free') = {p_spam_given_free:.4f}")

# Multiple evidence: "free" AND "win"
p_win_given_spam = 0.65; p_win_given_ham = 0.02
# Naive Bayes (conditional independence assumed)
p_spam_given_both = (p_free_given_spam * p_win_given_spam * p_spam)
p_ham_given_both  = (p_free_given_ham  * p_win_given_ham  * (1-p_spam))
p_spam_posterior  = p_spam_given_both / (p_spam_given_both + p_ham_given_both)
print(f"P(spam | 'free', 'win') = {p_spam_posterior:.4f}")